# GLiNER v1 multilingual baseline, per-label F1 report

Reads the persisted results from `ner/ner-evaluation/scores.csv` (the evaluator
appends one row per label + an `OVERALL` row each run) and reports the
**GLiNER v1 multilingual** baseline numbers, builds a per-label F1 matrix, and compares GliNER
against the LLM runs on the same 1816 letter.

Persisted artifacts are written to `GliNER_baseline/results/`:
- `gliner_per_label.csv` — full GliNER per-label P/R/F1 (strict + relaxed) + TP/FP/FN
- `gliner_f1_matrix.csv` — labels × {strict, relaxed} F1
- `gliner_f1_bar.png` — bar chart of the above
- `per_label_f1_comparison_relaxed.csv` / `.png` — labels × models relaxed-F1 heatmap

In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'GliNER_baseline' else Path('..').resolve()
SCORES = ROOT / 'ner' / 'ner-evaluation' / 'scores.csv'
OUT = (ROOT / 'GliNER_baseline' / 'results') if (ROOT / 'GliNER_baseline').exists() else Path('results')
OUT.mkdir(parents=True, exist_ok=True)
print('scores:', SCORES, '| exists:', SCORES.exists())
print('out dir:', OUT)

scores: /home/tijn-do/github/hist-dutch-travelogues-nlp/ner/ner-evaluation/scores.csv | exists: True
out dir: /home/tijn-do/github/hist-dutch-travelogues-nlp/GliNER_baseline/results


## 1. Load scores and isolate the GLiNER v1 multilingual run

In [2]:
df = pd.read_csv(SCORES)
# Numeric columns
num_cols = ['strict_p','strict_r','strict_f1','relaxed_p','relaxed_r','relaxed_f1',
            'count_tp','count_fp','count_fn','duration_seconds','temperature']
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

g = df[df['model'] == 'gliner_multi'].copy()
# Keep only the latest run if several exist (scores.csv is a cumulative log).
latest = g['datetime'].max()
g = g[g['datetime'] == latest].copy()
print('GliNER rows (latest run):', len(g), '| datetime:', latest)
g[['source_text','model','prompting_method','inference_type','datetime']].drop_duplicates()

GliNER rows (latest run): 9 | datetime: 2026-06-30 19:37:19


,source_text,model,prompting_method,inference_type,datetime
945,1816_third_letter.txt,gliner_multi,zeroshot,cpu,2026-06-30 19:37:19


## 2. GLiNER v1 multilingual per-label results (strict + relaxed)

In [3]:
label_order = [l for l in g['label'] if l != 'OVERALL'] + (['OVERALL'] if 'OVERALL' in set(g['label']) else [])
show = ['strict_p','strict_r','strict_f1','relaxed_p','relaxed_r','relaxed_f1','count_tp','count_fp','count_fn']
g_table = g.set_index('label').loc[label_order, show].round(3)
g_table.to_csv(OUT / 'gliner_per_label.csv')
g_table

,strict_p,strict_r,strict_f1,relaxed_p,relaxed_r,relaxed_f1,count_tp,count_fp,count_fn
label,,,,,,,,,
E18_Physical_Thing,0.000,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN
E19_Physical_Object,0.000,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN
E20_Biological_Object,0.000,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN
E31_Document,0.000,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN
E52_Time_Span,0.714,0.172,0.278,1.000,0.241,0.389,NaN,NaN,NaN
E53_Place,0.902,0.343,0.497,0.943,0.463,0.621,NaN,NaN,NaN
F2_Expression,0.000,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN
Mode_of_Transportation,0.000,0.000,0.000,0.000,0.000,0.000,NaN,NaN,NaN
OVERALL,0.857,0.134,0.231,0.934,0.181,0.303,44.0,4.0,271.0


## 3. Per-label F1 matrix (GLiNER v1 multilingual: strict vs relaxed)

In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

f1 = g.set_index('label').loc[label_order, ['strict_f1','relaxed_f1']].round(3)
f1.to_csv(OUT / 'gliner_f1_matrix.csv')

fig, ax = plt.subplots(figsize=(8, 5))
y = range(len(f1))
w = 0.4
ax.barh([y - w/2 for y in y], f1['strict_f1'], height=w, label='strict F1', color='#4c72b0')
ax.barh([y + w/2 for y in y], f1['relaxed_f1'], height=w, label='relaxed F1', color='#dd8452')
ax.set_yticks(list(y)); ax.set_yticklabels(f1.index)
ax.invert_yaxis()
ax.set_xlabel('F1'); ax.set_title('GLiNER v1 multilingual per-label F1 (1816 third letter)')
ax.legend(loc='lower right')
for yi, (s, r) in enumerate(zip(f1['strict_f1'], f1['relaxed_f1'])):
    ax.text(s + 0.01, yi - w/2, f'{s:.2f}', va='center', fontsize=8)
    ax.text(r + 0.01, yi + w/2, f'{r:.2f}', va='center', fontsize=8)
plt.tight_layout(); plt.savefig(OUT / 'gliner_f1_bar.png', dpi=130); plt.show()
f1

/tmp/ipykernel_15837/2110754341.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT / 'gliner_f1_bar.png', dpi=130); plt.show()


,strict_f1,relaxed_f1
label,,
E18_Physical_Thing,0.000,0.000
E19_Physical_Object,0.000,0.000
E20_Biological_Object,0.000,0.000
E31_Document,0.000,0.000
E52_Time_Span,0.278,0.389
E53_Place,0.497,0.621
F2_Expression,0.000,0.000
Mode_of_Transportation,0.000,0.000
OVERALL,0.231,0.303


## 4. Comparison: per-label relaxed F1 across all models

Rows = labels, columns = `model/prompting`. GLiNER v1 multilingual is the only `cpu`/`zeroshot`
column. Cells = relaxed F1 (any overlap counts as a match).

In [5]:
# Restrict to the same source text as the GliNER run, drop OVERALL for the heatmap.
src = g['source_text'].iloc[0]
cmp = df[df['source_text'] == src].copy()
cmp['run'] = cmp['model'] + '/' + cmp['prompting_method']
cmp = cmp[cmp['label'] != 'OVERALL']

heat = cmp.pivot_table(index='label', columns='run', values='relaxed_f1', aggfunc='max')
# Order columns: GliNER last so the eye lands on it.
cols = [c for c in heat.columns if not c.startswith('gliner')] + [c for c in heat.columns if c.startswith('gliner')]
heat = heat[cols]
heat.round(3).to_csv(OUT / 'per_label_f1_comparison_relaxed.csv')

import seaborn as sns
fig, ax = plt.subplots(figsize=(max(9, 0.55*len(heat.columns)), 5))
sns.heatmap(heat.round(2), annot=True, fmt='.2f', cmap='YlGnBu', ax=ax,
            cbar_kws={'label': 'relaxed F1'}, vmin=0)
ax.set_title(f'Per-label relaxed F1 — {src}')
plt.xticks(rotation=40, ha='right'); plt.tight_layout()
plt.savefig(OUT / 'per_label_f1_comparison_relaxed.png', dpi=130); plt.show()
heat.round(3)

/tmp/ipykernel_15837/779806424.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(OUT / 'per_label_f1_comparison_relaxed.png', dpi=130); plt.show()


run,cogito-2.1:671b/fewshot,cogito-2.1:671b/zeroshot,deepseek-v4-flash/fewshot,deepseek-v4-flash/zeroshot,deepseek-v4-pro/fewshot,deepseek-v4-pro/zeroshot,gemma4:31b/fewshot,gemma4:31b/zeroshot,glm-5.1/fewshot,glm-5.1/zeroshot,kimi-k2.7-code/fewshot,kimi-k2.7-code/zeroshot,mistral-large-3:675b/fewshot,mistral-large-3:675b/zeroshot,qwen3.5:397b/fewshot,qwen3.5:397b/zeroshot,gliner_multi/zeroshot
label,,,,,,,,,,,,,,,,,
E18_Physical_Thing,0.898,0.852,0.927,0.824,0.930,0.936,0.913,0.952,0.894,0.862,0.968,0.923,0.872,0.889,0.942,0.941,0.000
E19_Physical_Object,0.804,0.744,0.850,0.528,0.850,0.689,0.891,0.786,0.868,0.826,0.907,0.529,0.808,0.681,0.870,0.792,0.000
E20_Biological_Object,0.941,0.629,0.941,0.941,0.941,0.941,0.889,0.971,0.941,0.971,0.914,0.867,0.872,0.941,0.941,0.865,0.000
E31_Document,0.500,0.615,0.588,0.526,0.476,0.526,0.500,0.571,0.625,0.500,0.625,0.588,0.333,0.400,0.556,0.526,0.000
E52_Time_Span,0.947,0.537,0.983,0.949,0.951,0.949,0.931,0.900,0.906,0.966,0.892,0.936,0.912,0.807,0.906,0.915,0.389
E53_Place,0.940,0.860,0.967,0.962,0.972,0.972,0.938,0.938,0.934,0.958,0.972,0.967,0.935,0.932,0.959,0.968,0.621
F2_Expression,0.400,0.238,0.400,0.500,0.353,0.522,0.500,0.467,0.400,0.429,0.421,0.480,0.133,0.300,0.421,0.500,0.000
Mode_of_Transportation,0.750,0.410,0.929,0.824,0.885,0.815,0.881,0.897,0.889,0.857,0.897,0.897,0.776,0.889,0.873,0.815,0.000


## 5. GLiNER v1 multilingual summary

A short auto-generated narrative of where GLiNER v1 multilingual stands relative to the LLM runs.

In [6]:
overall = g.set_index('label').loc['OVERALL', ['strict_f1','relaxed_f1','count_tp','count_fp','count_fn']]
sub = df[df['source_text']==src].copy()
sub['run'] = sub['model'] + '/' + sub['prompting_method']
ov = sub[sub['label']=='OVERALL'].set_index('run')[['strict_f1','relaxed_f1']]
ov = ov[~ov.index.duplicated(keep='first')].sort_values('relaxed_f1', ascending=False).round(3)
print('OVERALL F1 by run (sorted by relaxed):')
print(ov.to_string())
print()
print(f"GLiNER v1 multilingual OVERALL: strict F1={overall['strict_f1']:.3f}, relaxed F1={overall['relaxed_f1']:.3f}")
print(f"  TP={int(overall['count_tp'])} FP={int(overall['count_fp'])} FN={int(overall['count_fn'])}")
llm = ov[ov.index.to_series().str.startswith('gliner') == False]
if not llm.empty:
    best = llm.iloc[0]
    print(f"Best LLM ({llm.index[0]}): strict F1={best['strict_f1']:.3f}, relaxed F1={best['relaxed_f1']:.3f}")
    print(f"Gap (relaxed): GLiNER v1 multilingual is {overall['relaxed_f1']-best['relaxed_f1']:+.3f} vs best LLM.")
ov.to_csv(OUT / 'overall_f1_by_run.csv')

OVERALL F1 by run (sorted by relaxed):
                               strict_f1  relaxed_f1
run                                                 
kimi-k2.7-code/fewshot             0.646       0.896
qwen3.5:397b/fewshot               0.648       0.870
deepseek-v4-pro/fewshot            0.656       0.868
deepseek-v4-flash/fewshot          0.574       0.864
gemma4:31b/fewshot                 0.643       0.861
qwen3.5:397b/zeroshot              0.608       0.848
gemma4:31b/zeroshot                0.612       0.846
deepseek-v4-pro/zeroshot           0.620       0.839
glm-5.1/fewshot                    0.602       0.826
glm-5.1/zeroshot                   0.549       0.813
mistral-large-3:675b/fewshot       0.604       0.809
kimi-k2.7-code/zeroshot            0.628       0.806
mistral-large-3:675b/zeroshot      0.569       0.776
deepseek-v4-flash/zeroshot         0.610       0.771
cogito-2.1:671b/fewshot            0.571       0.767
cogito-2.1:671b/zeroshot           0.368       0.626
gliner_